In [1]:
import glob, os, random
import numpy as np, pandas as pd
random.seed(0); np.random.seed(0)

# ---- 1. find all places ----
labs = sorted(glob.glob("/kaggle/input/**/labels.npz", recursive=True))
print("labels.npz found:", len(labs))
rows = []
for p in labs:
    d = os.path.dirname(p)
    run = os.path.basename(os.path.dirname(d))
    ok = emp = fail = -1
    f = os.path.join(d, "DONE")
    if os.path.exists(f):
        ok, emp, fail = [int(v) for v in open(f).read().strip().split(",")]
    rows.append(dict(run=run, id=int(os.path.basename(d)), dir=d, ok=ok, empty=emp, failed=fail))
inv = pd.DataFrame(rows)
print("\nplaces per run:\n", inv.groupby("run").size().to_string())
print("places without DONE file:", int((inv["ok"] < 0).sum()))
print("zero-month places:", inv[inv["ok"] == 0]["id"].tolist())
print("places with < 12 months:", int((inv["ok"] < 12).sum()), "| with 24 months:", int((inv["ok"] == 24).sum()))
print("months ok/empty/failed:", int(inv["ok"].sum()), int(inv["empty"].sum()), int(inv["failed"].sum()))

# ---- 2. label lookup tables -> 9 classes (253 = rangeland, 255 = unknown) ----
def lut(d):
    t = np.full(256, 255, np.uint8)
    for k, v in d.items(): t[k] = v
    return t
WC = lut({10:1, 20:5, 30:2, 40:4, 50:6, 60:7, 70:8, 80:0, 90:3, 95:1, 100:7})
ES = lut({1:0, 2:1, 4:3, 5:4, 7:6, 8:7, 9:8, 11:253})
DW = lut({i: i for i in range(9)})
LUTS = [WC, WC, ES, ES, DW, DW]
NAMES = ["wc2020", "wc2021", "esri2020", "esri2021", "dw2020", "dw2021"]
CLS = ["water", "trees", "grass", "flooded", "crops", "shrub", "built", "bare", "snow"]

def same(a, b):
    v = (a != 255) & (b != 255)
    low = lambda x: (x == 2) | (x == 5)
    eq = (a == b) | ((a == 253) & low(b)) | ((b == 253) & low(a))
    return v, eq & v

# ---- 3. sample 100 places per run and compare ----
samp = pd.concat([g.sample(min(100, len(g)), random_state=0) for _, g in inv.groupby("run")])
print("\nsampled places:", len(samp))
share = np.zeros((6, 256), np.float64)
pairs = [(1,3), (1,5), (3,5), (0,1), (2,3), (4,5)]
pn = ["wc21-esri21", "wc21-dw21", "esri21-dw21", "wc20-wc21", "esri20-esri21", "dw20-dw21"]
pv = np.zeros(6); pe = np.zeros(6)
tv = t2 = t3 = 0
for d in samp["dir"]:
    y = np.load(os.path.join(d, "labels.npz"))["y"]
    m = [LUTS[k][y[:, k]] for k in range(6)]
    for k in range(6): share[k] += np.bincount(m[k].ravel(), minlength=256)
    for i, (a, b) in enumerate(pairs):
        v, e = same(m[a], m[b]); pv[i] += v.sum(); pe[i] += e.sum()
    v1, e1 = same(m[1], m[3]); v2, e2 = same(m[1], m[5]); v3, e3 = same(m[3], m[5])
    allv = v1 & v2 & v3
    tv += allv.sum()
    t2 += ((e1 | e2 | e3) & allv).sum()
    t3 += ((e1 & e2 & e3) & allv).sum()

print("\n=== class share (%) of known pixels, per source ===")
tab = {}
for k in range(6):
    s = share[k]; known = s[:9].sum() + s[253]
    tab[NAMES[k]] = [100 * s[c] / known for c in range(9)] + [100 * s[253] / known, 100 * s[255] / s.sum()]
print(pd.DataFrame(tab, index=CLS + ["rangeland(esri)", "unknown%(of all)"]).round(1).to_string())

print("\n=== pixel agreement (%) ===")
for i in range(6): print(f"{pn[i]:15s} {100 * pe[i] / pv[i]:.1f}")
print(f"\n2021 triple: at least 2 of 3 agree = {100 * t2 / tv:.1f}% | all 3 agree = {100 * t3 / tv:.1f}% (pixels valid in all three: {tv:,})")

# ---- 4. Sentinel-1 sanity on 40 random month files ----
files = []
for d in samp["dir"]: files += glob.glob(os.path.join(d, "20??-??.npz"))
files = random.sample(files, min(40, len(files)))
bad = 0; vv = []; vh = []; zero = []
for f in files:
    x = np.load(f)["x"].astype(np.float32)
    bad += int(np.isnan(x).sum() + np.isinf(x).sum())
    zero.append(float((x == 0).mean()))
    vv.append(float(np.nanmean(x[:, 0]))); vh.append(float(np.nanmean(x[:, 1])))
print(f"\nS1 sanity on {len(files)} files: NaN/inf values={bad} | zero fraction mean={np.mean(zero):.4f} | mean VV={np.mean(vv):.2f} dB | mean VH={np.mean(vh):.2f} dB")

labels.npz found: 1124

places per run:
 run
v2_run1    225
v2_run2    225
v2_run3    225
v2_run4    225
v2_run5    224
places without DONE file: 0
zero-month places: [594, 632]
places with < 12 months: 8 | with 24 months: 1033
months ok/empty/failed: 26597 378 1

sampled places: 500

=== class share (%) of known pixels, per source ===
                  wc2020  wc2021  esri2020  esri2021  dw2020  dw2021
water                5.0     5.0       4.9       4.9     5.7     6.2
trees               34.8    35.7      32.4      32.9    30.5    31.6
grass               23.4    23.5       0.0       0.0     5.0     4.9
flooded              2.7     2.7       2.1       2.1     0.9     0.9
crops                9.2     9.3      10.0      10.2    10.7    10.9
shrub                6.9     6.3       0.0       0.0    12.3    12.4
built                3.8     4.2       6.5       6.5     5.8     5.9
bare                13.7    12.6       4.6       4.5    15.7    15.0
snow                 0.7     0.7       1.

In [2]:
cnt = dict(nan=0, pinf=0, ninf=0); bad_files = []; vvm = []; vhm = []; pct = []
for f in files:
    x = np.load(f)["x"].astype(np.float32)
    n, pi, ni = int(np.isnan(x).sum()), int(np.isposinf(x).sum()), int(np.isneginf(x).sum())
    cnt["nan"] += n; cnt["pinf"] += pi; cnt["ninf"] += ni
    if n + pi + ni: bad_files.append((os.path.relpath(f, "/kaggle/input"), n, pi, ni, x.size))
    a, b = x[:, 0], x[:, 1]
    vvm.append(a[np.isfinite(a)].mean()); vhm.append(b[np.isfinite(b)].mean())
    g = x[np.isfinite(x)]
    pct.append(np.percentile(g[::50], [0.1, 1, 50, 99, 99.9]))
print(cnt)
print("files with bad values:", len(bad_files), "of", len(files))
for b in bad_files: print(b)
print("finite mean VV, VH (dB):", round(float(np.mean(vvm)), 2), round(float(np.mean(vh_m := vhm)), 2))
print("percentiles 0.1/1/50/99/99.9 of finite values:", np.mean(pct, axis=0).round(1))

{'nan': 0, 'pinf': 0, 'ninf': 948}
files with bad values: 1 of 40
('notebooks/vinayybg/v2-run05/v2_run5/1049/2020-02.npz', 0, 0, 948, 1179648)
finite mean VV, VH (dB): -11.81 -19.08
percentiles 0.1/1/50/99/99.9 of finite values: [-28.4 -25.8 -15.3  -6.3  -3.7]
